# file_server_clean_up — Safe file audit & cleanup candidates

**What this notebook does**
- Recursively scans a target folder
- Flags obvious junk (OS temp/metadata, zero-byte files, etc.)
- Detects *content duplicates* using hashes (safe, filename-independent)
- Chooses one file to **KEEP** per duplicate set (configurable)
- Exports a **CSV** listing paths and files flagged as **DELETE_CANDIDATE** (no deletion by default)

**Safety**: This notebook does **not** delete anything unless you explicitly enable the optional quarantine step at the end.

## 0) Install checklist (run once)
You should already have created the conda environment and installed dependencies. See the repo README / setup steps from the chat.

In [1]:
import os
import sys
import math
import time
import hashlib
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm import tqdm

print('Python:', sys.version)
print('CWD:', os.getcwd())

Python: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]
CWD: c:\00_dev\file_server_clean_up\notebooks


## 1) Configuration (edit these)


In [2]:
# === REQUIRED: set the folder you want to scan ===
# Windows example:
# ROOT = Path(r"D:\\FILE_SERVER_EXPORT")
# Network share example:
# ROOT = Path(r"\\\\SERVER\\Share\\SomeFolder")

ROOT = Path(r"Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING")

# Where to write outputs
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exclusions
EXCLUDE_DIR_NAMES = {
    ".git", ".ipynb_checkpoints", "__pycache__", "node_modules",
    "$RECYCLE.BIN", "System Volume Information"
}

# Obvious junk filenames/extensions (add as needed)
JUNK_FILENAMES = {
    ".DS_Store", "Thumbs.db", "desktop.ini", "Icon\r", "._.DS_Store"
}
JUNK_EXTENSIONS = {
    ".tmp", ".temp", ".swp", ".swo", ".bak", ".old", ".dwl", ".dwl2", ".~lock"
}

# Duplicate detection / hashing
HASH_ALGO = "blake2b"   # blake2b is fast & in hashlib
FAST_HASH_BYTES = 2 * 1024 * 1024  # 2 MiB from head + 2 MiB from tail
FULL_HASH_FOR_FINAL_GROUPS = True  # set False to speed up but risk rare collisions

# How to pick the KEEP file within each duplicate group
# Options: "newest_mtime", "oldest_mtime", "shortest_path"
DUP_KEEP_STRATEGY = "newest_mtime"

# Optional directory preferences (higher wins). Example: prefer a 'CURRENT' folder.
# Provide as substrings (case-insensitive) or absolute paths.
PREFERRED_PATH_SUBSTRINGS = [
    # r"\\CURRENT\\",
    # r"\\_LATEST\\",
]

# Minimum file size (bytes) to consider for duplicate grouping
MIN_SIZE_FOR_DUP_CHECK = 1

# CSV name
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH = OUTPUT_DIR / f"cleanup_candidates_{RUN_TAG}.csv"

assert ROOT.exists(), f"ROOT does not exist: {ROOT}"

## 2) File scan (metadata only)


In [3]:
def iter_files(root: Path):
    # Conservative walk: ignore symlinks by default
    for dirpath, dirnames, filenames in os.walk(root):
        # prune excluded directories in-place
        dirnames[:] = [d for d in dirnames if d not in EXCLUDE_DIR_NAMES]
        for fn in filenames:
            p = Path(dirpath) / fn
            try:
                if p.is_symlink():
                    continue
                st = p.stat()
            except (FileNotFoundError, PermissionError, OSError):
                continue
            yield {
                "path": str(p),
                "name": p.name,
                "ext": p.suffix.lower(),
                "size": int(st.st_size),
                "mtime": float(st.st_mtime),
            }

rows = []
for rec in tqdm(iter_files(ROOT), desc="Scanning", unit="files"):
    rows.append(rec)

df = pd.DataFrame(rows)
df["mtime_iso"] = pd.to_datetime(df["mtime"], unit="s", utc=True).dt.tz_convert(None).astype(str)
df["is_zero_bytes"] = df["size"] == 0
df["is_junk_name"] = df["name"].isin(JUNK_FILENAMES)
df["is_junk_ext"] = df["ext"].isin(JUNK_EXTENSIONS)
df["junk_reason"] = ""
df.loc[df["is_zero_bytes"], "junk_reason"] += "zero_bytes;"
df.loc[df["is_junk_name"], "junk_reason"] += "junk_filename;"
df.loc[df["is_junk_ext"], "junk_reason"] += "junk_extension;"

df.shape, df.head()

Scanning: 1015files [03:52,  4.37files/s]


((1015, 10),
                                                 path  \
 0  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 1  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 2  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 3  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 4  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 
                                         name   ext     size         mtime  \
 0  SONADO_OITYLO_TOMH TOIXOY KAI TOIXIOY.dwg  .dwg   621200  1.667403e+09   
 1         01_SONADO_OITYLO_KATOPSH ST A'.pdf  .pdf  1865245  1.650031e+09   
 2         02_SONADO_OITYLO_KATOPSH ST B'.pdf  .pdf  4482305  1.650031e+09   
 3         03_SONADO_OITYLO_KATOPSH ST C'.pdf  .pdf  5285680  1.650031e+09   
 4         04_SONADO_OITYLO_KATOPSH ST D'.pdf  .pdf  4610105  1.650031e+09   
 
                        mtime_iso  is_zero_bytes  is_junk_name  is_junk_ext  \
 0  2022-11-02 15:31:59.000000000          False         False        False   
 1  2022-04-15 

## 3) Hashing helpers


In [4]:
def _new_hasher(algo: str):
    if algo == "blake2b":
        return hashlib.blake2b(digest_size=32)
    if algo == "sha256":
        return hashlib.sha256()
    raise ValueError(f"Unsupported algo: {algo}")

def fast_fingerprint(path: str, size: int, algo: str = HASH_ALGO, nbytes: int = FAST_HASH_BYTES) -> str:
    """Fast fingerprint: hash(first nbytes + last nbytes + size)."""
    h = _new_hasher(algo)
    h.update(str(size).encode("utf-8"))
    p = Path(path)
    with p.open("rb") as f:
        head = f.read(nbytes)
        h.update(head)
        if size > nbytes:
            # tail
            try:
                f.seek(max(0, size - nbytes))
                tail = f.read(nbytes)
                h.update(tail)
            except OSError:
                # some special files may not seek cleanly
                pass
    return h.hexdigest()

def full_hash(path: str, algo: str = HASH_ALGO, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = _new_hasher(algo)
    p = Path(path)
    with p.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

## 4) Duplicate detection (size → fast hash → full hash)


In [5]:
df["dup_eligible"] = df["size"] >= MIN_SIZE_FOR_DUP_CHECK

# Pass 1: only consider size groups with >1 files
size_counts = df.loc[df["dup_eligible"], "size"].value_counts()
candidate_sizes = set(size_counts[size_counts > 1].index.tolist())

cand = df[df["dup_eligible"] & df["size"].isin(candidate_sizes)].copy()
print("Duplicate candidates by size:", len(cand), "files across", len(candidate_sizes), "sizes")

# Pass 2: compute fast fingerprints for candidates
fast_hashes = []
for path, size in tqdm(zip(cand["path"], cand["size"]), total=len(cand), desc="Fast hashing"):
    try:
        fast_hashes.append(fast_fingerprint(path, int(size)))
    except (FileNotFoundError, PermissionError, OSError):
        fast_hashes.append(None)

cand["hash_fast"] = fast_hashes
cand = cand.dropna(subset=["hash_fast"]).copy()

# Pass 3: group by (size, hash_fast), and optionally full-hash within groups >1
cand["group_fast"] = cand.groupby(["size", "hash_fast"]).ngroup()
fast_group_sizes = cand["group_fast"].value_counts()
fast_dupe_groups = fast_group_sizes[fast_group_sizes > 1].index.tolist()

cand["hash_full"] = None
if FULL_HASH_FOR_FINAL_GROUPS and fast_dupe_groups:
    mask = cand["group_fast"].isin(fast_dupe_groups)
    sub = cand.loc[mask].copy()
    fulls = []
    for path in tqdm(sub["path"], total=len(sub), desc="Full hashing"):
        try:
            fulls.append(full_hash(path))
        except (FileNotFoundError, PermissionError, OSError):
            fulls.append(None)
    cand.loc[mask, "hash_full"] = fulls

# Determine the final duplicate key
cand["dup_key"] = cand.apply(
    lambda r: (r["size"], r["hash_full"]) if (FULL_HASH_FOR_FINAL_GROUPS and pd.notna(r["hash_full"])) else (r["size"], r["hash_fast"]),
    axis=1,
)

cand["dup_group_id"] = cand.groupby("dup_key").ngroup()
dup_group_counts = cand["dup_group_id"].value_counts()
dup_groups = dup_group_counts[dup_group_counts > 1].index.tolist()

print("Final duplicate groups:", len(dup_groups))
dup_group_counts.head(10)

Duplicate candidates by size: 842 files across 340 sizes


Full hashing: 100%|██████████| 830/830 [06:56<00:00,  1.99it/s]

Final duplicate groups: 339


dup_group_id
100    4
103    4
332    4
333    4
347    4
346    4
337    4
338    4
176    4
340    4
Name: count, dtype: int64

## 5) Decide KEEP vs DELETE_CANDIDATE


In [6]:
def preference_score(path: str) -> int:
    p = path.lower()
    score = 0
    for i, sub in enumerate(PREFERRED_PATH_SUBSTRINGS):
        if sub and sub.lower() in p:
            # earlier in list = higher weight
            score += (len(PREFERRED_PATH_SUBSTRINGS) - i) * 10_000
    # shorter path gets a tiny bump (helps tie-break)
    score += max(0, 500 - len(path))
    return score

cand["pref_score"] = cand["path"].apply(preference_score)

def pick_keeper(group: pd.DataFrame) -> int:
    if DUP_KEEP_STRATEGY == "newest_mtime":
        # preference score dominates, then newest mtime
        g = group.sort_values(["pref_score", "mtime"], ascending=[False, False])
    elif DUP_KEEP_STRATEGY == "oldest_mtime":
        g = group.sort_values(["pref_score", "mtime"], ascending=[False, True])
    elif DUP_KEEP_STRATEGY == "shortest_path":
        g = group.assign(path_len=group["path"].str.len()).sort_values(["pref_score", "path_len"], ascending=[False, True])
    else:
        raise ValueError(f"Unknown strategy: {DUP_KEEP_STRATEGY}")
    return g.index[0]

# Mark keepers within each duplicate group
cand["dup_is_keeper"] = False
keepers = []
for gid, g in tqdm(cand[cand["dup_group_id"].isin(dup_groups)].groupby("dup_group_id"), desc="Choosing keepers"):
    keep_idx = pick_keeper(g)
    keepers.append(keep_idx)
cand.loc[keepers, "dup_is_keeper"] = True

# Merge duplicate info back to full df
df = df.merge(
    cand[["path", "hash_fast", "hash_full", "dup_group_id", "dup_is_keeper"]],
    on="path", how="left"
)

# Decision logic (conservative):
# - If junk => DELETE_CANDIDATE
# - Else if in duplicate group and not keeper => DELETE_CANDIDATE
# - Else => KEEP
df["decision"] = "KEEP"
df["reason"] = "";
df.loc[df["junk_reason"] != "", "decision"] = "DELETE_CANDIDATE"
df.loc[df["junk_reason"] != "", "reason"] += df.loc[df["junk_reason"] != "", "junk_reason"]

dup_nonkeepers = df["dup_group_id"].notna() & (df["dup_is_keeper"] == False)
df.loc[dup_nonkeepers & (df["decision"] == "KEEP"), "decision"] = "DELETE_CANDIDATE"
df.loc[dup_nonkeepers & (df["decision"] == "DELETE_CANDIDATE"), "reason"] += "duplicate_nonkeeper;"
df.loc[dup_nonkeepers & (df["decision"] == "KEEP"), "reason"] += "duplicate_nonkeeper;"

# For keepers in a dupe group, note it
df.loc[df["dup_group_id"].notna() & (df["dup_is_keeper"] == True), "reason"] += "duplicate_keeper;"

# Clean reason formatting
df["reason"] = df["reason"].str.strip(";")

df["decision"].value_counts(), df.head()

Choosing keepers: 100%|██████████| 339/339 [00:00<00:00, 3324.35it/s]


(decision
 DELETE_CANDIDATE    527
 KEEP                488
 Name: count, dtype: int64,
                                                 path  \
 0  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 1  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 2  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 3  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 4  Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...   
 
                                         name   ext     size         mtime  \
 0  SONADO_OITYLO_TOMH TOIXOY KAI TOIXIOY.dwg  .dwg   621200  1.667403e+09   
 1         01_SONADO_OITYLO_KATOPSH ST A'.pdf  .pdf  1865245  1.650031e+09   
 2         02_SONADO_OITYLO_KATOPSH ST B'.pdf  .pdf  4482305  1.650031e+09   
 3         03_SONADO_OITYLO_KATOPSH ST C'.pdf  .pdf  5285680  1.650031e+09   
 4         04_SONADO_OITYLO_KATOPSH ST D'.pdf  .pdf  4610105  1.650031e+09   
 
                        mtime_iso  is_zero_bytes  is_junk_name  is_junk_ext  \
 0  2022-11-02 15:3

## 6) Export CSV (paths + flags)


In [7]:
out = df[[
    "path", "decision", "reason", "size", "mtime_iso",
    "ext", "hash_fast", "hash_full", "dup_group_id", "dup_is_keeper"
]].copy()

out.to_csv(CSV_PATH, index=False, encoding="utf-8")
print("Wrote:", CSV_PATH)
out["decision"].value_counts()

Wrote: ..\outputs\cleanup_candidates_20260302_141826.csv


decision
DELETE_CANDIDATE    527
KEEP                488
Name: count, dtype: int64

## 7) Review helpers


In [8]:
# Preview what would be deleted
out[out["decision"] == "DELETE_CANDIDATE"].head(50)

,path,decision,reason,size,mtime_iso,ext,hash_fast,hash_full,dup_group_id,dup_is_keeper
35,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_filename,201728,2026-03-02 06:15:04.759999990,.db,NaN,NaN,NaN,NaN
36,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,664636,2022-11-02 15:31:59.000000000,.bak,055c707cbb87860279fe4c5d0c54e25f469745f4667040...,68e798b735d59dc43921dfd2f747dee588f7c8089ba6e8...,100.0,False
37,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,692091,2022-11-02 15:31:59.000000000,.dwg,e807257c06127f08c623938fb92bd9912d64e8aea54406...,98680a7abb133af8e4a5982793a7fd3d755f8ddab3b38a...,103.0,False
38,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,9554920,2022-11-02 15:31:59.000000000,.bak,fa7831c4357200d18b7a05593acf8267f6dd61e09aec30...,c5f40c6bcf021969264cd1a41f9649d621196426e3fd76...,332.0,False
39,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,9598848,2022-11-02 15:31:59.000000000,.dwg,038f8df44a1ab0393616d561ebba2d31887edd94d2a899...,26ea00678608e44154dc5aa8808566c719c4837a6bd1d7...,333.0,False
40,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,False
41,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,25864399,2022-11-02 15:31:59.000000000,.dwg,39f3f0e383d75266718cc953ea381f7f3549366440f084...,6ad4e54ec9e9255e0831849cebf97260796907108cbde3...,346.0,False
42,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,14532003,2022-11-02 15:31:59.000000000,.bak,6cbf074e5d49b8b24b1c6201390416d929bcfe5ed02596...,f6689a35b8ddf1cb3c55d13adcfab6394b1f5ea57786b5...,337.0,False
43,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,14532198,2022-11-02 15:31:59.000000000,.dwg,22f771bf30cd1ced37a60ee949b7a4c52953f822d9ad7e...,292b5fd7480f9e9734b035eaec274b6b4daa2a0cd5706d...,338.0,False
47,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_keeper,664636,2022-11-02 15:31:59.000000000,.bak,055c707cbb87860279fe4c5d0c54e25f469745f4667040...,68e798b735d59dc43921dfd2f747dee588f7c8089ba6e8...,100.0,True


In [9]:
# Counts by reason (for DELETE_CANDIDATE rows)
tmp = out[out['decision'] == 'DELETE_CANDIDATE'].copy()
tmp['reason_list'] = tmp['reason'].fillna('').apply(lambda s: [r for r in s.split(';') if r])
reason_counts = (
    tmp.explode('reason_list')
       .query("reason_list != ''")
       .groupby('reason_list')
       .size()
       .sort_values(ascending=False)
)
reason_counts.head(50)


reason_list
duplicate_nonkeeper    503
junk_extension          39
junk_filename           20
duplicate_keeper        11
dtype: int64

In [10]:
# Largest delete candidates (often duplicates)
out[out["decision"] == "DELETE_CANDIDATE"].sort_values("size", ascending=False).head(50)

,path,decision,reason,size,mtime_iso,ext,hash_fast,hash_full,dup_group_id,dup_is_keeper
974,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,28062159,2022-11-02 15:31:59.000000000,.bak,0c9c733d7b975278214c02ab89567faca1f5960e1f46f4...,5b3727d7e1f6b61acacf45d44f5fee7cb246d4538add23...,350.0,False
55,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_keeper,28062159,2022-11-02 15:31:59.000000000,.bak,0c9c733d7b975278214c02ab89567faca1f5960e1f46f4...,5b3727d7e1f6b61acacf45d44f5fee7cb246d4538add23...,350.0,True
650,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,28062159,2022-11-02 15:31:59.000000000,.bak,0c9c733d7b975278214c02ab89567faca1f5960e1f46f4...,5b3727d7e1f6b61acacf45d44f5fee7cb246d4538add23...,350.0,False
975,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27783962,2022-11-02 15:31:59.000000000,.dwg,896982e43dd1ec115dac5669a29362d5c5f8721c97af75...,d98a7ac2f56dabc05853f4f124949754d88626f3a11547...,349.0,False
651,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27783962,2022-11-02 15:31:59.000000000,.dwg,896982e43dd1ec115dac5669a29362d5c5f8721c97af75...,d98a7ac2f56dabc05853f4f124949754d88626f3a11547...,349.0,False
680,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27562520,2022-11-02 15:31:59.000000000,.pdf,2b7d6ab5499c268b2bc64b1d150241132091dce3098032...,bb898e194bb2a0b83ddf8d7ad6bbffc4eb0f2e559f0894...,348.0,False
1005,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,duplicate_nonkeeper,27562520,2022-11-02 15:31:59.000000000,.pdf,2b7d6ab5499c268b2bc64b1d150241132091dce3098032...,bb898e194bb2a0b83ddf8d7ad6bbffc4eb0f2e559f0894...,348.0,False
646,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,False
40,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_nonkeeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,False
51,Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_...,DELETE_CANDIDATE,junk_extension;duplicate_keeper,26347123,2022-11-02 15:31:59.000000000,.bak,4bd5483f21da657324d6e1bc37eae60ca8317525786854...,e4403d8df4dc956d35e48fe244ee0e9f9d0b1faec6b80e...,347.0,True


## 8) OPTIONAL: Quarantine move (safer than delete)
**Default is disabled.** If you enable it, files are moved to a quarantine folder preserving relative paths.

⚠️ Only do this after reviewing the CSV.


In [ ]:
ENABLE_QUARANTINE = True
QUARANTINE_DIR = Path("..") / "quarantine" / f"run_{RUN_TAG}"

def move_to_quarantine(src_path: str, root: Path, quarantine_root: Path):
    src = Path(src_path)
    rel = src.relative_to(root)
    dst = quarantine_root / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists():
        raise FileExistsError(f"Destination already exists: {dst}")

    try:
        src.replace(dst)  # atomic move on same volume
    except OSError:
        # Cross-volume fallback (e.g., Z: -> C:)
        import shutil
        shutil.move(str(src), str(dst))

if ENABLE_QUARANTINE:
    QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)
    to_move = out[out["decision"] == "DELETE_CANDIDATE"]["path"].tolist()
    moved = 0
    for p in tqdm(to_move, desc="Quarantining"):
        try:
            move_to_quarantine(p, ROOT, QUARANTINE_DIR)
            moved += 1
        except Exception as e:
            print("FAILED:", p, "->", e)
    print(f"Moved {moved}/{len(to_move)} files to {QUARANTINE_DIR}")
else:
    print("Quarantine is disabled (ENABLE_QUARANTINE=False).")

Quarantining:   0%|          | 2/527 [00:00<00:53,  9.74it/s]

FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\Thumbs.db -> [WinError 17] The system cannot move the file to a different disk drive: 'Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\Thumbs.db' -> '..\\quarantine\\run_20260302_141826\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\Thumbs.db'
FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\01_SONADO_OITYLO_KATOPSH A STATHMIS.bak -> [WinError 17] The system cannot move the file to a different disk drive: 'Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\\01_SONADO_OITYLO_KATOPSH A STATHMIS.bak' -> '..\\quarantine\\run_20260302_141826\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\\01_SONADO_OITYLO_KATOPSH A STATHMIS.bak'


Quarantining:   1%|          | 4/527 [00:00<00:54,  9.61it/s]

FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\01_SONADO_OITYLO_KATOPSH A STATHMIS.dwg -> [WinError 17] The system cannot move the file to a different disk drive: 'Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\\01_SONADO_OITYLO_KATOPSH A STATHMIS.dwg' -> '..\\quarantine\\run_20260302_141826\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\\01_SONADO_OITYLO_KATOPSH A STATHMIS.dwg'
FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\02_SONADO_OITYLO_KATOPSH B STATHMIS.bak -> [WinError 17] The system cannot move the file to a different disk drive: 'Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\24 ΑΡΧΙΤΕΚΤΟΝΙΚΗ ΜΕΛΕΤΗ\\DWG ΑΡΧΙΤΕΚΤΟΝΙΚΑ\\02_SONADO_OITYLO_KATOPSH B STATHMIS.bak' -> '..\\

Quarantining:   1%|          | 6/527 [00:00<00:53,  9.69it/s]

...

FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\PDF ΣΤΑΤΙΚΑ\ΣΤ3-02  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ ΘΕΜΕΛΙΩΣΗΣ - ΥΠΟΓΕΙΟΥ.pdf -> [WinError 17] The system cannot move the file to a different disk drive: 'Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\\PDF ΣΤΑΤΙΚΑ\\ΣΤ3-02  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ ΘΕΜΕΛΙΩΣΗΣ - ΥΠΟΓΕΙΟΥ.pdf' -> '..\\quarantine\\run_20260302_141826\\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\\PDF ΣΤΑΤΙΚΑ\\ΣΤ3-02  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ ΘΕΜΕΛΙΩΣΗΣ - ΥΠΟΓΕΙΟΥ.pdf'
FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\PDF ΣΤΑΤΙΚΑ\ΣΤ3-03  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ Α' ΣΤΑΘΜΗΣ.pdf -> [WinError 17] The system cannot move the file to a different disk drive: "Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OI

Quarantining: 100%|█████████▉| 525/527 [00:54<00:00,  9.74it/s]

FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\PDF ΣΤΑΤΙΚΑ\ΣΤ3-04  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ Β', Γ' ΣΤΑΘΜΗΣ.pdf -> [WinError 17] The system cannot move the file to a different disk drive: "Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\\PDF ΣΤΑΤΙΚΑ\\ΣΤ3-04  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ Β', Γ' ΣΤΑΘΜΗΣ.pdf" -> "..\\quarantine\\run_20260302_141826\\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\\PDF ΣΤΑΤΙΚΑ\\ΣΤ3-04  - ΚΤΗΡΙΟ 3 ΞΥΛΟΤΥΠΟΣ Β', Γ' ΣΤΑΘΜΗΣ.pdf"
FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\PDF ΣΤΑΤΙΚΑ\ΣΤ4-01  - ΚΤΗΡΙΟ 4 ΞΥΛΟΤΥΠΟΣ ΘΕΜΕΛΙΩΣΗΣ Α' ΣΤΑΘΜΗΣ.pdf -> [WinError 17] The system cannot move the file to a different disk drive: "Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKA

Quarantining: 100%|██████████| 527/527 [00:54<00:00,  9.60it/s]

FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\PDF ΣΤΑΤΙΚΑ\ΣΤ4-02  - ΚΤΗΡΙΟ 4 ΞΥΛΟΤΥΠΟΣ Β', Γ' ΣΤΑΘΜΗΣ.pdf -> [WinError 17] The system cannot move the file to a different disk drive: "Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOKKALA_MANI\\04_DESIGN_ENGINEERING\\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\\PDF ΣΤΑΤΙΚΑ\\ΣΤ4-02  - ΚΤΗΡΙΟ 4 ΞΥΛΟΤΥΠΟΣ Β', Γ' ΣΤΑΘΜΗΣ.pdf" -> "..\\quarantine\\run_20260302_141826\\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\\PDF ΣΤΑΤΙΚΑ\\ΣΤ4-02  - ΚΤΗΡΙΟ 4 ΞΥΛΟΤΥΠΟΣ Β', Γ' ΣΤΑΘΜΗΣ.pdf"
FAILED: Z:\SONADO-IK-801455240\10_PROJECTS\HTL0049-01_OITYLO-KOKKALA_MANI\04_DESIGN_ENGINEERING\ΤΕΛΙΚΗ ΜΕΛΕΤΗ ΑΡΧΙΤΕΚΤΟΝΙΚΑ_ΣΤΑΤΙΚΑ\ΣΤΑΤΙΚΗ ΜΕΛΕΤΗ\PDF ΣΤΑΤΙΚΑ\ΣΤ5-01  - ΒΙΟΛΟΓΙΚΟΣ ΚΑΘΑΡΙΣΜΟΣ & ΔΕΞΑΜΕΝΗ ΞΥΛΟΤΥΠΟΙ.pdf -> [WinError 17] The system cannot move the file to a different disk drive: 'Z:\\SONADO-IK-801455240\\10_PROJECTS\\HTL0049-01_OITYLO-KOK